### Middlewares

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased Summarization
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer =  InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
        model="google_genai:gemini-3.1-flash-lite",
        trigger=("messages", 6),
        keep = ("messages", 3))
    ]
)

In [8]:
## Run with thread id

config = {"configurable": {"thread_id":"test-1"}}

In [10]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"messages: {response}")
    print(f"Messages: {len(response['messages'])}")

messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='e52cc311-9b52-4b29-b996-acdae3167401'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'EmAKXgFpFH0T0SktZ9qtwDzidcwc2bPXDaQvoUdjOi48CHmfzab+xx6WwKet2mEu5YTRqfYdiXokDgJPFXbMaJK92p4Fy9E+ur3BA3Ok6qC6C648/PQzxbHogF6HIuwWJ7o='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e071-cdbc-77e9-9f2e-f84857ac0b73-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a543472d-1923-4897-b012-353ce752687f'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 equals 4.', 'extras': {'signature': 'EmAKXgFpFH0TOr6zGDBsqQN0CJsQ/dtkgvQl6XBiMZgdjb

### Token Size

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model = "google_genai:gemini-3.1-flash-lite",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "google_genai:gemini-3.1-flash-lite",
            trigger = ("tokens", 350),
            keep = ("tokens", 150)
        )
    ]
)

config = {"configurable": {"thread_id":"test-1"}}

# Token Counter Approximate
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars ~ 1 token


In [16]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~163 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='b1fa6145-74ee-4dec-844c-67235c7c1b6b'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_8924298606966535779': 'EnMKcQFpFH0TKulzPcrj/Ns8u9lEh9KBMOaOV2e2tW2vZsVoUGBuMdne1BGA+v0v7Ge+WlfxUAuRn38FrHG2pxivofn42bZ1Br6k+CT0rQa2ruAem2fHyS4j+9FVS4qXn5Yrsx5MU3nr4YVRYgAVnq+GI3o9'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e07f-cc67-77a8-a051-71373ef3f40b-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_8924298606966535779', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}), ToolMessage(content

### Fraction


In [ ]:
## Same as before just mentioning tokens in Fraction

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str)-> str:
    """ Mock Function to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
     """Mock function to send an email"""
     return f"Email sent to {recipient} with subject {subject}"

In [20]:
agent = create_agent(
    model = "google_genai:gemini-3.1-flash-lite",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool": {
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [21]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='ef77e2ef-3212-440e-9e25-65e5a70144ba'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "body": "How are you?", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_15738303777561300278': 'EnMKcQFpFH0TBriMyurPbysayuMNAEcxQ7wMuW1li8HhVwouio9eppihsIwTvg4fxIFwLrvyT+WOeCm0CMLBwZbP+sUlsbvlORMYbUCL2LYJ//ZY8rAJiqc8tmpbm5uEKfBBQ4TK7LA44S6WkI542cDUums3'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e096-2fd7-7577-bcc1-4601e0ca5da2-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'body': 'How are you?', 'recipient': 'john@test.com'}, 'id': 'call_15738303777561300278', 'type': 'tool_call'}], invalid

#### Approve

In [23]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "OK. I've sent the email to john@test.com.", 'extras': {'signature': 'EnMKcQFpFH0TeQJ4vB2exbjyrINn5RQaqYYUOhpdzW7MqGe7NbtSyRgq69s77oGbi4lNTvp1seLiL/UwbKMDMF42R2SNAsQZe+pRXI/xM7NS/xM/2B/Pn66yO3b1yPX712KWX5pb6DaggJ9FJh+MWZeAUILh'}}]


In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='ef77e2ef-3212-440e-9e25-65e5a70144ba'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "body": "How are you?", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_15738303777561300278': 'EnMKcQFpFH0TBriMyurPbysayuMNAEcxQ7wMuW1li8HhVwouio9eppihsIwTvg4fxIFwLrvyT+WOeCm0CMLBwZbP+sUlsbvlORMYbUCL2LYJ//ZY8rAJiqc8tmpbm5uEKfBBQ4TK7LA44S6WkI542cDUums3'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e096-2fd7-7577-bcc1-4601e0ca5da2-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'body': 'How are you?', 'recipient': 'john@test.com'}, 'id': 'call_15738303777561300278', 'type': 'tool_call'}], invalid

#### Reject

In [25]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [26]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'The email was not sent because the request was rejected. Please let me know if you would like me to proceed with sending it.', 'extras': {'signature': 'EnMKcQFpFH0TgjIqZ9Ol3W5pF6FhTpMGtLA/BjIrjGd9Ej9svAX5pJq7wcYhGt+g20g/Lo68hQX6Nld6VJs0vY8YeVFpd7pODrK0v3dH34wpxK8XmLDG+otGpXhwJAt1ViUmRHrR4xZFcZuIKTw6dEZLO8gt'}}]


In [27]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='db44357d-fcac-4d4d-801a-de9211d2e2e5'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_7425267981157362530': 'EnMKcQFpFH0TVLJtUTB4kQpqe0705TAjMDuWAdRpXjy00mwIHArBYP79KutgcC7Jwj3PQmpK2t8m4i/JjveTKXmSOVJuCUT7Nna2pMTB74DK6bzswF7mYW9wa2JuydSfIqKFTQsOlp12n4qccClRGrjXLDlu'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e09a-7aab-70d9-ac9b-bf9fb6764dfc-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'call_7425267981157362530', 'type': 'tool_call'}], invalid_t

#### Editing

In [33]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)


result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='6c288dae-7020-4cc1-afa9-3ce9cdf27737'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "wrong@email.com", "body": "Hello", "subject": "Test"}'}, '__gemini_function_call_thought_signatures__': {'call_5010778475501859379': 'EnMKcQFpFH0ThxNRR/CVS0QMbfVE4avtqptHaMYWtMlNAfpw3UgFVnW5Ox3NnDpRBE0ZSEyTxnAwxf3kRW6HXyf9w4Z9Vx8vt8MLfDAGl9qUR7tU78HywfepqAoCcqws/H4k8anGfDNLZpScL72mMLsbKk4G'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e09b-b728-7595-88eb-aa5cd4f37fb8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@email.com', 'body': 'Hello', 'subject': 'Test'}, 'id': 'call_5010778475501859379', 'type': 'tool_call'}], invalid_tool_calls=[], usag

In [34]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: [{'type': 'text', 'text': "The email was not sent to 'wrong@email.com'. As with your previous request, a human reviewer intercepted and modified it. The email was sent to 'correct@email.com' with the subject 'Corrected Subject' and the body 'This was edited by human before sending'.", 'extras': {'signature': 'EnMKcQFpFH0TQ42/623gZQ9jHr0/L13JJ8Y+9z1mrllx5Gh9lDGHIyjl3NIGPHYHpWnzwaQTUXlETj+XByXIaFSXgAcQf8q+O+D/vEvMeaNLOSa9OR2vN+DC9+whHGgT4knCOJrdjOZpevzxFrduDcmIVB1t'}}]


In [36]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='6c288dae-7020-4cc1-afa9-3ce9cdf27737'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "wrong@email.com", "body": "Hello", "subject": "Test"}'}, '__gemini_function_call_thought_signatures__': {'call_5010778475501859379': 'EnMKcQFpFH0ThxNRR/CVS0QMbfVE4avtqptHaMYWtMlNAfpw3UgFVnW5Ox3NnDpRBE0ZSEyTxnAwxf3kRW6HXyf9w4Z9Vx8vt8MLfDAGl9qUR7tU78HywfepqAoCcqws/H4k8anGfDNLZpScL72mMLsbKk4G'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e09b-b728-7595-88eb-aa5cd4f37fb8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@email.com', 'body': 'Hello', 'subject': 'Test'}, 'id': 'call_5010778475501859379', 'type': 'tool_call'}], invalid_tool_calls=[], usag